# 03 Explain & Results

Shows feature group importance, top-10 action cards, and 3 sample HIGH-risk item explanations.

## Feature Group Importance

In [1]:
import json, pandas as pd
with open('../models/importance.json') as f: imp = json.load(f)
for g in sorted(imp, key=lambda x: x['importance'], reverse=True):
    bar = '█' * int(g['importance'] * 400)
    print(f"  {g['label']:<45} {g['importance']:.4f}  {bar}")


  Inventory Coverage & Stock Buffer             0.2255  ██████████████████████████████████████████████████████████████████████████████████████████
  Store Assortment & Product Characteristics    0.0311  ████████████
  Recent Demand Momentum & Rolling Trends       0.0219  ████████
  Upcoming Weekend & Holiday Signals            0.0038  █
  Price Discount & Active Promotions            0.0010  
  Local Weather Variations                      0.0010  


## Top-10 Action Cards (by Priority Score)

In [2]:
preds = pd.read_parquet('../data/processed/predictions.parquet')
top10 = preds.sort_values('priority_score', ascending=False).head(10)
cols = ['store_id','product_id','category','stockout_prob','risk','reorder_qty','priority_score','confidence']
print(top10[cols].to_string(index=False))


store_id product_id  category  stockout_prob   risk  reorder_qty  priority_score confidence
     S02       P106     Dairy         0.9073   HIGH          295        54894.86       high
     S02       P132 Groceries         0.7652   HIGH          204        49524.93       high
     S01       P132 Groceries         0.8794   HIGH          226        47567.50       high
     S01       P106     Dairy         0.9179   HIGH          223        39118.00       high
     S05       P132 Groceries         0.8065   HIGH          104        33023.18       high
     S03       P106     Dairy         0.8864   HIGH          221        31642.57       high
     S05       P106     Dairy         0.8287   HIGH          258        31635.95       high
     S03       P132 Groceries         0.8423   HIGH          168        30702.80       high
     S02       P112 Beverages         0.9383   HIGH          118        30053.49       high
     S02       P110 Beverages         0.6814 MEDIUM          283        29953.29

## 3 Sample HIGH-Risk Explanations

In [3]:
high3 = preds[preds['risk']=='HIGH'].sort_values('priority_score', ascending=False).head(3)
for _, row in high3.iterrows():
    print(f"\n{row['store_id']}-{row['product_id']} ({row['category']}) | prob={row['stockout_prob']:.2%} | order+{row['reorder_qty']}")
    for r in row['reasons']:
        print(f"  • {r}")



S02-P106 (Dairy) | prob=90.73% | order+295
  • Low stock cover (2.1 days) (100%)

S02-P132 (Groceries) | prob=76.52% | order+204
  • No single dominant driver (100%)

S01-P132 (Groceries) | prob=87.94% | order+226
  • Low stock cover (1.4 days) (58%)
  • Below reorder level (4 units) (42%)


## Limitations

- **Synthetic data**: generated by `src/make_dataset.py` (seed 42). Real NovaMart data would replace this.
- **incoming_stock = 0**: in-transit pipeline inventory not observed.
- **Weekend flag**: constant over any 7-day window — cannot be a per-row driver.
- **Lost-sales gap**: baseline estimate is 29.3% below simulated truth (rolling-mean underestimates on promo/weekend spikes).
- **SVM skipped**: too slow on >10k rows.
- **Sparse-history products**: P138, P139, P140 use category-store mean fallback (confidence = low).
- **Target underestimation**: `next_7_day_demand` is built from sales, which are capped by stock on stock-out days.